In [51]:
%run 00_chargement_donnees.ipynb

In [52]:
# Volume par année
vol_annee = df_all.groupby(['ANNEE', 'ST']).size().reset_index(name='count')
vol_annee['%'] = vol_annee.groupby('ANNEE')['count'].transform(lambda x: x / x.sum() * 100).round(1)

# Volume par mois
vol_mois = df_all.groupby(['ANNEE', 'MOIS', 'ST']).size().reset_index(name='count')
vol_mois['%'] = vol_mois.groupby(['ANNEE', 'MOIS'])['count'].transform(lambda x: x / x.sum() * 100).round(1)

> ⚠️ **Note : filtre CLIENT sur 2024 non fiable**
> Pour l'année 2024, la correspondance OTP → Client est une estimation basée sur les numéros de train disponibles. La donnée client précise n'était pas renseignée à l'époque. Les résultats filtrés par client sur 2024 sont donc à interpréter avec prudence.

## Tableau récapitulatif — Prestations par ST et par année

In [53]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

clients = ['TOUS', 'TX MECA NORD', 'GRAND PROJET', 'DR SUD', 'CHAILLOUE', 'ETF DIRECTION MATERIEL', 'EXTERNE']
filtre_client_recap = widgets.Dropdown(options=clients, value='TOUS', description='Client :')

def afficher_recap(client):
    df_f = df_all if client == 'TOUS' else df_all[df_all['CLIENT'] == client]
    vol = df_f.groupby(['ANNEE', 'ST']).size().reset_index(name='count')

    recap = vol.pivot_table(index='ST', columns='ANNEE', values='count', aggfunc='sum').fillna(0).astype(int)
    recap.columns = [str(c) for c in recap.columns]

    for col in ['2024', '2025', '2026']:
        if col not in recap.columns:
            recap[col] = 0

    def evol(a, b):
        with np.errstate(divide='ignore', invalid='ignore'):
            return np.where(a == 0, None, ((b - a) / a * 100).round(1))

    recap['Évol. 24→25 (%)'] = evol(recap['2024'].values, recap['2025'].values)
    recap['Évol. 25→26 (%)'] = evol(recap['2025'].values, recap['2026'].values)
    recap = recap[['2024', '2025', '2026', 'Évol. 24→25 (%)', 'Évol. 25→26 (%)']].sort_values('2025', ascending=False)

    # Séparer INTERNE des ST externes
    interne = recap.loc[recap.index == 'INTERNE'] if 'INTERNE' in recap.index else pd.DataFrame()
    externe = recap.loc[recap.index != 'INTERNE']

    total_ext = pd.DataFrame({
        '2024': [externe['2024'].sum()],
        '2025': [externe['2025'].sum()],
        '2026': [externe['2026'].sum()],
        'Évol. 24→25 (%)': evol(externe['2024'].sum(), externe['2025'].sum()),
        'Évol. 25→26 (%)': evol(externe['2025'].sum(), externe['2026'].sum()),
    }, index=['TOTAL ST EXTERNE'])

    total_global = pd.DataFrame({
        '2024': [recap['2024'].sum()],
        '2025': [recap['2025'].sum()],
        '2026': [recap['2026'].sum()],
        'Évol. 24→25 (%)': evol(recap['2024'].sum(), recap['2025'].sum()),
        'Évol. 25→26 (%)': evol(recap['2025'].sum(), recap['2026'].sum()),
    }, index=['TOTAL GLOBAL'])

    display(pd.concat([externe, total_ext, interne, total_global]))

widgets.interact(afficher_recap, client=filtre_client_recap)

interactive(children=(Dropdown(description='Client :', options=('TOUS', 'TX MECA NORD', 'GRAND PROJET', 'DR SU…

<function __main__.afficher_recap(client)>

## Répartition ST (camembert)

In [54]:
mois_options = [(nom, i+1) for i, nom in enumerate(noms_mois)]

filtre_annee = widgets.Dropdown(options=[2024, 2025, 2026], value=2025, description='Année :')
filtre_debut = widgets.Dropdown(options=mois_options, value=1, description='Début :')
filtre_fin = widgets.Dropdown(options=mois_options, value=12, description='Fin :')
filtre_client_cam = widgets.Dropdown(options=clients, value='TOUS', description='Client :')

def on_annee_change_cam(change):
    if change['new'] == 2024:
        filtre_client_cam.value = 'TOUS'
        filtre_client_cam.disabled = True
    else:
        filtre_client_cam.disabled = False

filtre_annee.observe(on_annee_change_cam, names='value')

def afficher_camembert(annee, mois_debut, mois_fin, client):
    SEUIL = 3
    df_f = df_all[
        (df_all['ANNEE'] == annee) &
        (df_all['MOIS'] >= mois_debut) &
        (df_all['MOIS'] <= mois_fin)
    ]
    if client != 'TOUS':
        df_f = df_f[df_f['CLIENT'] == client]

    vol = df_f.groupby('ST').size().reset_index(name='count')
    vol['%'] = (vol['count'] / vol['count'].sum() * 100).round(1)
    vol.loc[vol['%'] < SEUIL, 'ST'] = 'AUTRES'
    vol = vol.groupby('ST')[['count', '%']].sum().reset_index()

    nom_debut = noms_mois[mois_debut - 1]
    nom_fin = noms_mois[mois_fin - 1]

    fig = px.pie(vol, names='ST', values='count',
                 title=f'Répartition ST — {annee} ({nom_debut} à {nom_fin})')
    fig.update_traces(texttemplate='%{label}<br>%{percent:.1%} (%{value})')
    fig.show()

widgets.interact(afficher_camembert, annee=filtre_annee, mois_debut=filtre_debut, mois_fin=filtre_fin, client=filtre_client_cam)

interactive(children=(Dropdown(description='Année :', index=1, options=(2024, 2025, 2026), value=2025), Dropdo…

<function __main__.afficher_camembert(annee, mois_debut, mois_fin, client)>

## Top 5 ST — Comparaison par année

In [55]:
filtre_client_top5 = widgets.Dropdown(options=clients, value='TOUS', description='Client :')

def afficher_top5(client):
    df_f = df_all if client == 'TOUS' else df_all[df_all['CLIENT'] == client]

    top5 = (
        df_f.groupby('ST').size()
        .sort_values(ascending=False)
        .head(5)
        .index.tolist()
    )

    vol = df_f.groupby(['ANNEE', 'ST']).size().reset_index(name='count')
    df_top5 = vol[vol['ST'].isin(top5)]

    fig = px.bar(
        df_top5, x='ST', y='count', color='ANNEE', barmode='group',
        title='Top 5 ST — Nombre de prestations par année',
        labels={'count': 'Nombre de prestations', 'ANNEE': 'Année'},
        color_discrete_sequence=px.colors.qualitative.Set2,
    )
    fig.show()

widgets.interact(afficher_top5, client=filtre_client_top5)

interactive(children=(Dropdown(description='Client :', options=('TOUS', 'TX MECA NORD', 'GRAND PROJET', 'DR SU…

<function __main__.afficher_top5(client)>

## Volume mensuel par ST

In [56]:
st_liste = sorted(df_all['ST'].dropna().unique())
dropdown_st = widgets.Dropdown(options=st_liste, description='ST :')
dropdown_annee = widgets.Dropdown(options=[2024, 2025, 2026], value=2025, description='Année :')
filtre_client_mensuel = widgets.Dropdown(options=clients, value='TOUS', description='Client :')

def on_annee_change_mensuel(change):
    if change['new'] == 2024:
        filtre_client_mensuel.value = 'TOUS'
        filtre_client_mensuel.disabled = True
    else:
        filtre_client_mensuel.disabled = False

dropdown_annee.observe(on_annee_change_mensuel, names='value')

def afficher_mensuel(st, annee, client):
    df_f = df_all[
        (df_all['ST'] == st) & (df_all['ANNEE'] == annee)
    ].dropna(subset=['MOIS']).copy()

    if client != 'TOUS':
        df_f = df_f[df_f['CLIENT'] == client]

    vol = df_f.groupby('MOIS').size().reset_index(name='Nombre de prestations')
    vol['MOIS'] = vol['MOIS'].astype(int)
    vol['PERIODE'] = vol['MOIS'].apply(lambda m: noms_mois[m - 1])

    fig = px.bar(vol, x='PERIODE', y='Nombre de prestations',
                 title=f'Volume mensuel — {st} — {annee}',
                 category_orders={'PERIODE': noms_mois},
                 text='Nombre de prestations')
    fig.update_traces(textposition='outside')
    fig.show()

widgets.interact(afficher_mensuel, st=dropdown_st, annee=dropdown_annee, client=filtre_client_mensuel)

interactive(children=(Dropdown(description='ST :', options=('???', 'ATEAM', 'CLAISSERAIL', 'CLMTP', 'COLAS', '…

<function __main__.afficher_mensuel(st, annee, client)>